In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd '/content/drive/MyDrive/groundwork'
!pip install -r requirements.txt -q

In [ ]:
# Run data pipeline for all cities. Takes ~2-3 hours total.
# Skip cities already complete by checking data/ directory.
!python data_pipeline/cdg.py --config data_pipeline/cities.yaml --output data/

In [ ]:
!python model/train_vae.py \
    --data data/ \
    --output checkpoints/vae/ \
    --epochs 50 \
    --batch 4

In [ ]:
import torch, numpy as np, matplotlib.pyplot as plt
from model.vae import RoadVAE
from data_pipeline.dataset import RoadLayoutDataset

model = RoadVAE()
ckpt = torch.load('checkpoints/vae/vae_epoch_050.pth', map_location='cpu')
model.load_state_dict(ckpt['model'])
model.eval()

ds = RoadLayoutDataset(['data/irving_tx'], augment=False)
_, road = ds[0]
with torch.no_grad():
    recon, _, _ = model(road.unsqueeze(0))
recon_argmax = recon[0].argmax(0).numpy()
road_argmax  = road.argmax(0).numpy()

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(road_argmax,  cmap='tab10', vmin=0, vmax=4); axes[0].set_title('Original')
axes[1].imshow(recon_argmax, cmap='tab10', vmin=0, vmax=4); axes[1].set_title('Reconstructed')
plt.show()
# Target: visually similar, SSIM > 0.70